# 05 – Analysis dataset for ARM

**Purpose:** Merge the processed tables and turn them into **transactions** that association rules can be run on. A transaction is one municipality in a given year. Each transaction contains a number of *items*, for example `kommungrupp=C`, `bnp_indikator=svag` and `fruktsamhet=hög`.

| | |
|---|---|
| **Input** | `data/processed/fertilitet_kommun.csv` (02), `befolkningstathet.csv` (03), `konjunktur_ar.csv` (04) |
| **Output** | `data/processed/arm_underlag_lag1.csv`, `data/processed/arm_underlag_lag2.csv` |
| **Dependencies** | Run `01`–`04` first. |

### Research question

> *In which types of municipality is fertility high or low compared with other municipalities, when the business cycle the year before was weak or strong?*

The question is not *"does a recession lower fertility?"*. The business cycle is **national**, so all municipalities have the same business cycle position in a given year. That gives only 26 independent observations, and ARM is the wrong tool for so few. The outcome `skillnad_mot_riket` has also already had the national trend removed. What the data *can* answer is whether the municipality types **differ from each other** when the business cycle changes.

### Decisions implemented
| Decision | Choice |
|---|---|
| Outcome | `skillnad_mot_riket` in tertiles: *låg / medel / hög* (low / medium / high) |
| Business cycle indicators | GDP indicator, household consumption, employment |
| Business cycle classes | *svag* (weak, ≤ −0.5) / *normal* / *stark* (strong, > 0.5) |
| Municipality type | SKR's A/B/C |
| Lag | 1 year in the main analysis, 2 years as a sensitivity analysis |
| Training / test | Class thresholds are derived on 2003–2015 and tested on 2016–2025 |

## 1. Settings and parameters

All choices from the table above are collected here, so that they can be changed in one place.

- `FORSTA_AR = 2003`: first outcome year. With a 2-year lag, 2003 needs the business cycle from 2001. *Sysselsättning* (employment) is missing for 2000 (the series starts in 2001). Starting in 2003 makes the period **the same for both lags**, so that the results can be compared.
- `TRANING` / `TEST`: the early period is used to derive class thresholds and rules. The later period is only used to test them.

In [1]:
from pathlib import Path

import pandas as pd

# The notebook is normally run from the Assignment folder.
# If it is run from the project root, we locate Assignment anyway.
BAS = Path.cwd()
if not (BAS / "data" / "raw").is_dir() and (BAS / "Assignment").is_dir():
    BAS = BAS / "Assignment"

RAW = BAS / "data" / "raw"
PROCESSED = BAS / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

assert RAW.is_dir(), "Cannot find data/raw. Open the notebook from the Assignment folder."

INDIKATORER = ["bnp_indikator", "hushallens_konsumtion", "sysselsattning"]
LAGAR = [1, 2]
KONJ_GRANS = 0.5

FORSTA_AR = 2003
SISTA_AR = 2025
TRANING = (2003, 2015)
TEST = (2016, 2025)

## 2. Load the processed tables

`Kommunkod` is read as text, so that leading zeros are kept. We check that the keys are unique. Each join in the next step can then be checked with `validate`.

In [2]:
fert = pd.read_csv(PROCESSED / "fertilitet_kommun.csv", encoding="utf-8-sig", dtype={"Kommunkod": str})
tathet = pd.read_csv(PROCESSED / "befolkningstathet.csv", encoding="utf-8-sig", dtype={"Kommunkod": str})
konj = pd.read_csv(PROCESSED / "konjunktur_ar.csv", encoding="utf-8-sig")

assert not fert.duplicated(["Kommunkod", "ar"]).any()
assert not tathet.duplicated(["Kommunkod", "ar"]).any()
assert not konj.duplicated("ar").any()
assert set(INDIKATORER) <= set(konj.columns)

print("Fertility:", fert.shape, "| Density:", tathet.shape, "| Business cycle:", konj.shape)

Fertility: (7358, 9) | Density: (7358, 9) | Business cycle: (26, 14)


## 3. Join fertility and municipality type

Fertility and density have the same municipalities and years (this was checked in `03`), so the join should be **one-to-one**. From the density table only the municipality group is needed, since that is what is used as an item. We also check that the municipality names match on every row.

In [3]:
bas = fert.merge(
    tathet[["Kommunkod", "ar", "Kommun", "kommungrupp"]],
    on=["Kommunkod", "ar"], how="inner", validate="one_to_one", suffixes=("", "_tathet"),
)
assert len(bas) == len(fert)
assert (bas["Kommun"] == bas["Kommun_tathet"]).all()
bas = bas.drop(columns="Kommun_tathet")

print("Rows after join:", len(bas))

Rows after join: 7358


## 4. Restrict the period and drop rows without an outcome

Two filters, and the number of rows removed by each is reported:

1. **The period 2003–2025** (see step 1).
2. **Rows where fertility is missing.** A transaction without an outcome cannot contribute to a rule about fertility. Missing data is already handled and flagged in `02`. Here we only remove the rows where the value is still missing.

In [4]:
fore = len(bas)
bas = bas.loc[bas["ar"].between(FORSTA_AR, SISTA_AR)]
efter_period = len(bas)
bas = bas.loc[bas["skillnad_mot_riket"].notna()].reset_index(drop=True)

print(f"Outside the period {FORSTA_AR}–{SISTA_AR}: {fore - efter_period} rows")
print(f"Missing fertility within the period: {efter_period - len(bas)} rows")
print(f"Remaining: {len(bas)} municipality-years in {bas['Kommunkod'].nunique()} municipalities")

Outside the period 2003–2025: 849 rows
Missing fertility within the period: 42 rows
Remaining: 6467 municipality-years in 283 municipalities


## 5. Split into training and test

The split is made **in time**, not at random. A random split would put municipalities from the same year in both training and test. The business cycle is the same for all municipalities in a given year, so the test set would then consist of years the model has already seen.

With a time split, the class thresholds and rules are derived on 2003–2015 and tested on years the model has never seen.

In [5]:
bas["del"] = "test"
bas.loc[bas["ar"].between(*TRANING), "del"] = "traning"
assert bas.loc[bas["del"] == "test", "ar"].between(*TEST).all()

print(bas.groupby("del").agg(kommun_ar=("ar", "size"), forsta_ar=("ar", "min"), sista_ar=("ar", "max")))

         kommun_ar  forsta_ar  sista_ar
del                                    
test          2804       2016      2025
traning       3663       2003      2015


## 6. Classify fertility into tertiles

`skillnad_mot_riket` is split into three equally sized classes: *låg* (low), *medel* (medium) and *hög* (high). Equally sized classes ensure that each class has enough support to produce rules.

**The thresholds are computed on the training years only** and are then applied unchanged to the test years. 

**About the names:** the classes are relative to **other municipalities**, not to the national rate. The median of `skillnad_mot_riket` is about +0.12 (see `02`), so even the class *låg* contains municipalities slightly above the national rate. *Hög* therefore means "among the third of municipalities that are highest", not "above the national rate". 

In [6]:
FRUKT_KLASSER = ["låg", "medel", "hög"]

gransvarden = bas.loc[bas["del"] == "traning", "skillnad_mot_riket"].quantile([1 / 3, 2 / 3]).tolist()
bas["fruktsamhet_klass"] = pd.cut(
    bas["skillnad_mot_riket"],
    bins=[-float("inf"), *gransvarden, float("inf")],
    labels=FRUKT_KLASSER,
).astype(str)

print(f"Thresholds (from the training years): låg ≤ {gransvarden[0]:+.3f} < medel ≤ {gransvarden[1]:+.3f} < hög")
display(pd.crosstab(bas["del"], bas["fruktsamhet_klass"], normalize="index")[FRUKT_KLASSER].round(3))

Thresholds (from the training years): låg ≤ +0.030 < medel ≤ +0.210 < hög


fruktsamhet_klass,låg,medel,hög
del,,,
test,0.287,0.351,0.362
traning,0.338,0.331,0.332


## 7. Attach the business cycle with a lag

Fertility in a given year is linked to the business cycle **one (or two) years earlier**. A child born in year *t* was conceived about nine months earlier, and the decision to have a child was made even earlier. The business cycle position in the year of birth can therefore hardly have influenced it.

Technically, the years of the business cycle table are shifted forward by `lag` years (`ar + lag`), and the tables are then joined on year. The join is **many-to-one**: many municipalities get the same business cycle value in the same year.

Both lags are built here. One gives the main analysis and the other the sensitivity analysis.

In [7]:
underlag = {}
for lag in LAGAR:
    forskjuten = konj[["ar", *INDIKATORER]].assign(ar=konj["ar"] + lag)
    df = bas.merge(forskjuten, on="ar", how="left", validate="many_to_one")
    assert df[INDIKATORER].notna().all().all(), f"Business cycle data missing for some year with lag {lag}."
    underlag[lag] = df
    print(f"Lag {lag}: fertility {df['ar'].min()}–{df['ar'].max()} linked to business cycle "
          f"{df['ar'].min() - lag}–{df['ar'].max() - lag}, {len(df)} rows")

Lag 1: fertility 2003–2025 linked to business cycle 2002–2024, 6467 rows
Lag 2: fertility 2003–2025 linked to business cycle 2001–2023, 6467 rows


## 8. Classify the business cycle

The business cycle position is already on a standardised scale where 0 is the normal level (see `04`). Therefore **fixed thresholds** are used, which do not need to be derived from the data:

- *svag* (weak): ≤ −0.5
- *normal*: between −0.5 and +0.5
- *stark* (strong): > +0.5

The tables show **how many different years** fall into each class. This is the most important measure of how much data lies behind a rule. A business cycle class that only occurs in two years gives only two independent observations, even though it covers thousands of municipality-years. 

In [8]:
KONJ_KLASSER = ["svag", "normal", "stark"]

for lag, df in underlag.items():
    for ind in INDIKATORER:
        df[f"{ind}_klass"] = pd.cut(
            df[ind], bins=[-float("inf"), -KONJ_GRANS, KONJ_GRANS, float("inf")], labels=KONJ_KLASSER,
        ).astype(str)

    antal_ar = pd.DataFrame({
        ind: df.drop_duplicates("ar").groupby(["del", f"{ind}_klass"]).size() for ind in INDIKATORER
    }).fillna(0).astype(int)
    print(f"\nLag {lag}: number of years per business cycle class")
    display(antal_ar)


Lag 1: number of years per business cycle class


bnp_indikator  hushallens_konsumtion  sysselsattning
del                                                                 
test    normal              6                      2               3
        stark               3                      6               5
        svag                1                      2               2
traning normal              5                      4               5
        stark               3                      2               3
        svag                5                      7               5


Lag 2: number of years per business cycle class


bnp_indikator  hushallens_konsumtion  sysselsattning
del                                                                 
test    normal              6                      2               3
        stark               3                      6               5
        svag                1                      2               2
traning normal              6                      5               4
        stark               3                      2               4
        svag                4                      6               5

## 9. Final check and save

For each lag we check that:

- the key `Kommunkod` + `ar` is unique
- no items are missing
- all classes have one of the allowed values.

The files contain both the **classes** (what ARM uses) and the **numeric values** they are based on. That way every classification can be checked afterwards. The flags from `02` (`interpolerad`, `har_bortfall`) are carried along for the sensitivity analysis in `06`.

In [9]:
KOLUMNER = ["Kommunkod", "Kommun", "ar", "del", "kommungrupp",
            "skillnad_mot_riket", "fruktsamhet_klass",
            *INDIKATORER, *[f"{ind}_klass" for ind in INDIKATORER],
            "interpolerad", "har_bortfall"]

for lag, df in underlag.items():
    df = df[KOLUMNER].sort_values(["Kommunkod", "ar"]).reset_index(drop=True)
    klass_kolumner = ["kommungrupp", "fruktsamhet_klass", *[f"{ind}_klass" for ind in INDIKATORER]]

    assert not df.duplicated(["Kommunkod", "ar"]).any()
    assert df[klass_kolumner].notna().all().all()
    assert set(df["fruktsamhet_klass"]) <= set(FRUKT_KLASSER)
    assert set(df["kommungrupp"]) <= {"A", "B", "C"}
    for ind in INDIKATORER:
        assert set(df[f"{ind}_klass"]) <= set(KONJ_KLASSER)

    utfil = PROCESSED / f"arm_underlag_lag{lag}.csv"
    df.to_csv(utfil, index=False, encoding="utf-8-sig")
    print(f"Saved {len(df)} rows to {utfil.relative_to(BAS)}")

Saved 6467 rows to data/processed/arm_underlag_lag1.csv
Saved 6467 rows to data/processed/arm_underlag_lag2.csv
